<a href="https://www.kaggle.com/code/gedebhayuadhipramana/1-titanic-survival-prediction?scriptVersionId=339909879" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import kagglehub 

# download latest version
path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/titanic


In [2]:
from pathlib import Path
import pandas as pd 

# 1. Path Configuration
dataset_path = kagglehub.competition_download('titanic')
data_dir = Path(dataset_path)

train_path = data_dir / "train.csv"
test_path = data_dir / "test.csv"

# 2. Directory Inspection
print("Dataset contents:")
for file in data_dir.iterdir():
    print(f"- {file.name}")

# 3. Data ingestion
print("\nLoading datasets...")
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# 4. Data Verification
print(f"Train shape : {df_train.shape}")
print(f"Test shape  : {df_test.shape}")

#5. Data Profiling 
print("\nTrain preview:")
display(df_train.head(3))

Dataset contents:
- train.csv
- test.csv
- gender_submission.csv

Loading datasets...
Train shape : (891, 12)
Test shape  : (418, 11)

Train preview:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [3]:
# 1. Data Profiling

print("Missing values in Train data:")
missing_train = df_train.isna().sum()
display(missing_train[missing_train > 0])


print("\nMissing values in Test data:")
missing_test = df_test.isna().sum()
display(missing_test[missing_test > 0])

Missing values in Train data:


Age         177
Cabin       687
Embarked      2
dtype: int64


Missing values in Test data:


Age       86
Fare       1
Cabin    327
dtype: int64

In [4]:
# 1. Drop 'Cabin' safely (Idempotent approach)
df_train = df_train.drop(columns=['Cabin'], errors='ignore')
df_test = df_test.drop(columns=['Cabin'], errors='ignore')

# 2. Impute numerical columns with Median
df_train['Age'] = df_train['Age'].fillna(df_train['Age'].median())
df_test['Age'] = df_test['Age'].fillna(df_train['Age'].median())
df_test['Fare'] = df_test['Fare'].fillna(df_train['Fare'].median())

# 3. Impute categorical column with Mode
mode_embarked = df_train['Embarked'].mode()[0]
df_train['Embarked'] = df_train['Embarked'].fillna(mode_embarked)

# 4. Final Verification
print("Missing values after imputation:")
print(f"Train : {df_train.isna().sum().sum()}")
print(f"Test  : {df_test.isna().sum().sum()}")

Missing values after imputation:
Train : 0
Test  : 0


In [5]:
# 1. Binary Encoding for 'Sex'
df_train['Sex'] = df_train['Sex'].map({'male': 0, 'female': 1})
df_test['Sex'] = df_test['Sex'].map({'male': 0, 'female': 1})

# 2. One-Hot Encoding for 'Embarked'
df_train = pd.get_dummies(df_train, columns=['Embarked'], dtype=int)
df_test = pd.get_dummies(df_test, columns=['Embarked'], dtype=int)

# 3. Drop non-predictive text features
drop_cols = ['Name', 'Ticket', 'PassengerId']

# 4. Define Features (X) and Target (y)
X_train = df_train.drop(columns=drop_cols + ['Survived'], errors='ignore')
y_train = df_train['Survived']

# Save test IDs for final submission, then drop columns
test_ids = df_test['PassengerId']
X_test = df_test.drop(columns=drop_cols, errors='ignore')

# 5. Verification
print("Feature Engineering completed.")
print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")

print("\nFinal Training Features Preview:")
display(X_train.head(3))

Feature Engineering completed.
X_train shape : (891, 9)
X_test shape  : (418, 9)

Final Training Features Preview:


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,3,0,22.0,1,0,7.2500,0,0,1
1,1,1,38.0,1,0,71.2833,1,0,0
2,3,1,26.0,0,0,7.9250,0,0,1


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Local Validation Split (80% Train, 20% Validation)
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# 2. Model Initialization
# n_estimators: Jumlah pohon, max_depth: Kedalaman logika pohon
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# 3. Model Training Phase
print("Training the Random Forest model...")
model.fit(X_train_split, y_train_split)

# 4. Local Prediction & Evaluation
y_pred_val = model.predict(X_val)
local_accuracy = accuracy_score(y_val, y_pred_val)

print(f"Local Validation Accuracy : {local_accuracy:.4f}")

Training the Random Forest model...
Local Validation Accuracy : 0.8156


In [7]:
# 1. Final Training (Train on 100% of available training data)
print("Retraining model on full dataset for maximum performance...")
model.fit(X_train, y_train)

# 2. Final Prediction on Unknown Data
print("Predicting test data...")
final_predictions = model.predict(X_test)

# 3. Build Submission DataFrame
submission_df = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': final_predictions
})

# 4. Export to CSV (index=False ensures no extra row numbers are added)
submission_file = 'submission.csv'
submission_df.to_csv(submission_file, index=False)

# 5. Final Verification
print(f"\nSuccess! '{submission_file}' generated with shape: {submission_df.shape}")
display(submission_df.head())

Retraining model on full dataset for maximum performance...
Predicting test data...

Success! 'submission.csv' generated with shape: (418, 2)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
